# Databricks Auto Loader

# 1. What is Auto Loader?

Auto Loader is a Databricks feature used to incrementally ingest new files from cloud storage.

## It:

- Monitors a cloud storage location.
- Detects new files arriving in that location.
- Processes only new files incrementally.
- Works with Spark Structured Streaming.
- Supports file formats such as CSV, JSON and Parquet.
- Is designed for scalable file ingestion.

## Simple Example

**Suppose new sales files arrive every hour:**

```text
sales/
├── sales_10AM.csv
├── sales_11AM.csv
├── sales_12PM.csv
└── sales_1PM.csv    ← New file
```

Auto Loader detects and processes the new file.

## Interview Answer

**Auto Loader is a Databricks feature used to efficiently and incrementally ingest new files from cloud storage using Spark Structured Streaming.**

---

# 2. How Auto Loader Works

```text
Cloud Storage
      ↓
New files arrive
      ↓
Auto Loader detects new files
      ↓
readStream
      ↓
Streaming DataFrame
      ↓
Transformations
      ↓
writeStream
      ↓
Delta Table
```

## Step-by-Step

### Step 1: Specify the source

Tell Auto Loader where files arrive.

```text
/source/sales/
```

### Step 2: Specify the file format

```python
.option("cloudFiles.format", "csv")
```

### Step 3: Auto Loader detects new files

It identifies new files that need to be processed.

### Step 4: Read the files

```python
df = spark.readStream...
```

The data is loaded into a streaming DataFrame.

### Step 5: Transform the data

Example:

```python
df = df.filter("amount > 0")
```

### Step 6: Write the data

```python
df.writeStream...
```

### Step 7: Checkpoint tracks streaming progress

If the stream stops and restarts, the checkpoint helps it recover from its previous progress.

---

# 3. Basic Auto Loader Code

```python
df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .load("/source/sales/")
)
```

## Line-by-Line Explanation

### `spark.readStream`

**Means:**

Read the data as a stream, rather than a one-time batch.

### `.format("cloudFiles")`

**Means:**

Use Auto Loader.

### `.option("cloudFiles.format", "csv")`

**Means:**

Incoming files are CSV files.

**For JSON:**

```python
.option("cloudFiles.format", "json")
```

**For Parquet:**

```python
.option("cloudFiles.format", "parquet")
```

### `.load("/source/sales/")`

**Means:**

Read files arriving in this source location.

---

# 4. What is readStream?

`readStream` is part of Spark Structured Streaming.

It is used to read data continuously/incrementally from a streaming source.

## Auto Loader

```python
spark.readStream \
    .format("cloudFiles")
```

## Kafka

```python
spark.readStream \
    .format("kafka")
```

## Delta

```python
spark.readStream \
    .format("delta")
```

## Easy to Remember

```text
readStream
     ↓
Spark Streaming API

cloudFiles
     ↓
Auto Loader
```

---

# 5. readStream vs Auto Loader

This is important for interviews.

They are not two completely separate alternatives.

Auto Loader itself uses `readStream`.

```python
spark.readStream \
    .format("cloudFiles")
```

## Simple Understanding

| Code | Purpose |
|---|---|
| `readStream` | Start a streaming read |
| `.format("cloudFiles")` | Use Auto Loader |
| `cloudFiles.format` | Specify CSV/JSON/Parquet etc. |
| `.load()` | Specify source location |

## Interview Answer

### What is the difference between readStream and Auto Loader?

**readStream** is the Spark Structured Streaming API used to read streaming data.

**Auto Loader** is a Databricks file-ingestion feature used with `readStream` by specifying:

```python
.format("cloudFiles")
```

---

# 6. What is a Checkpoint?

Think of a checkpoint as a **bookmark for your streaming job**.

## Example

```text
Batch 1 → Processed
Batch 2 → Processed
Batch 3 → Processed
Batch 4 → Processing
             ↓
           Failure
```

When the stream starts again:

```text
Checkpoint
     ↓
Checks previous progress
     ↓
Stream recovers
     ↓
Continues processing
```

## Code

```python
.option(
    "checkpointLocation",
    "/checkpoint/sales"
)
```

## Interview Answer

**A checkpoint stores the progress and state of a streaming query so that it can recover after a failure or restart.**

## Why is Checkpoint Important?

- Helps recover after failure.
- Maintains streaming progress.
- Helps avoid reprocessing already committed data.
- Provides fault tolerance.

---

# 7. writeStream

After Auto Loader reads the data:

```python
df = spark.readStream...
```

We can write the streaming data using:

```python
df.writeStream
```

## Example

```python
query = (
    df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/checkpoint/sales"
        )
        .start("/target/sales")
)
```

## Line-by-Line Explanation

### 1. `df.writeStream`

```python
df.writeStream
```

**Means:**

Write the streaming DataFrame.

### 2. `.format("delta")`

```python
.format("delta")
```

**Means:**

Write the output in Delta format.

### 3. `.outputMode("append")`

```python
.outputMode("append")
```

**Means:**

Add new output rows to the target.

### 4. `.option("checkpointLocation", ...)`

```python
.option(
    "checkpointLocation",
    "/checkpoint/sales"
)
```

**Means:**

Store the streaming query's progress and recovery information.

### 5. `.start()`

```python
.start("/target/sales")
```

**Means:**

Start the streaming query and write the data to the target location.

### Databricks Auto Loader Internals and State Management

# Auto Loader Final Master Table

> **Use this table as your one-page revision sheet.**

| Component | What is it? | Why does it exist? | Stores | Created When | Used When | Question it Answers | If Missing | Real Life Analogy |
|---|---|---|---|---|---|---|---|---|
| **Schema Location** | Schema repository | Avoid schema inference every restart and support schema evolution | Column names, datatypes, schema versions, schema history | First schema inference | Stream start, schema evolution | What does my data look like? | Schema must be inferred again; schema evolution becomes difficult | Building blueprint |
| **RocksDB** | Embedded Key-Value Database | Track millions/billions of files efficiently | File path, file metadata, discovery status, processed status | During file discovery | Every trigger | Have I already processed this file? | Files may be reprocessed; scalability issues | Attendance register |
| **Micro Batch** | Processing unit | Process files in groups instead of one by one | Files belonging to a trigger | Every trigger | During execution | What files am I processing now? | No execution unit | Truck carrying packages |
| **Offsets** | Input log | Record planned work before execution | Batch input details | Before batch execution | Recovery | What should this batch process? | Spark won't know pending work | To-do list |
| **Commits** | Success log | Record completed work | Successful batch IDs | After successful execution | Recovery | What completed successfully? | Spark can't determine completed batches | Completed task list |
| **Sources** | Source metadata | Remember source configuration | Input path, source type, format | Stream initialization | Restart | Where is data coming from? | Source recovery becomes difficult | Address of warehouse |
| **Checkpoint** | Recovery package | Recover stream after failure | Offsets, commits, sources, metadata, RocksDB state | Stream start | Restart/Failure | How do I resume after failure? | Stream starts from scratch | Brain memory |

---

# Auto Loader Maintains 3 Types of State

| State Type | Purpose | Component |
|---|---|---|
| **Schema State** | Structure of data | Schema Location |
| **File State** | File processing history | RocksDB |
| **Execution State** | Batch execution history | Offsets + Commits + Checkpoint |

---

# Checkpoint Internals

```text
checkpoint/
├── offsets/
├── commits/
├── sources/
├── metadata
└── rocksdb/
```

| Folder | Purpose | Example |
|---|---|---|
| **offsets** | Planned batches | Batch 10 should process file20, file21 |
| **commits** | Successful batches | Batch 10 completed |
| **sources** | Source information | cloudFiles, ADLS path |
| **metadata** | Query details | Stream ID |
| **rocksdb** | File tracking database | file20 processed |

---

# File Processing Flow

### Assume

```text
file1.json
file2.json
file3.json
```

arrive in **ADLS**.

| Step | Component | Action |
|---|---|---|
| **1** | Auto Loader | Discovers files |
| **2** | RocksDB | Checks if files were seen earlier |
| **3** | RocksDB | Registers new files |
| **4** | Micro Batch | Creates Batch 0 |
| **5** | Offsets | Stores Batch 0 input plan |
| **6** | Schema Location | Loads schema |
| **7** | Spark | Reads file contents |
| **8** | Delta | Writes records |
| **9** | Commits | Marks Batch 0 successful |
| **10** | RocksDB | Updates file status to processed |
| **11** | Checkpoint | Saves stream state |

---

# Offset vs Commit

> **Most Important Interview Question**

| Feature | Offset | Commit |
|---|---|---|
| **Meaning** | Planned Work | Completed Work |
| **Created** | Before processing | After processing |
| **Indicates Success** | No | Yes |
| **Represents** | Input | Completion |
| **Used For** | Recovery planning | Recovery confirmation |
| **Example** | Process file1, file2 | Batch completed successfully |

## Easy Memory

> **Offset = I WILL process**  
> **Commit = I HAVE processed**

---

# Failure Scenario

### Suppose

### Before Crash

**Offsets**

```text
0
1
2
3
4
5
```

**Commits**

```text
0
1
2
3
4
```

### Analysis

| Item | Status |
|---|---|
| **Offset 5** | Exists |
| **Commit 5** | Missing |

### Meaning

```text
Batch 5 started
Batch 5 did not finish
```

### Restart Action

> **Re-run Batch 5**

This prevents data loss.

---

# RocksDB Deep Understanding

## Conceptual View

| Key | Value |
|---|---|
| file1.json | Processed |
| file2.json | Processed |
| file3.json | Processed |

---

# New File Detection

### Storage

```text
file1
file2
file3
file4
```

| File | RocksDB Result | Action |
|---|---|---|
| **file1** | Found | Skip |
| **file2** | Found | Skip |
| **file3** | Found | Skip |
| **file4** | Not Found | Process |

---

# Why RocksDB?

| Without RocksDB | With RocksDB |
|---|---|
| Millions of filenames in driver memory | Stored efficiently on disk |
| High memory consumption | Low memory usage |
| Poor scalability | Handles billions of files |
| Risk of duplicates | Exactly-once processing |

---

# Schema Evolution Example

## Version 1

```json
{
  "id": 1,
  "name": "John"
}
```

| Column | Type |
|---|---|
| id | bigint |
| name | string |

## Version 2

```json
{
  "id": 1,
  "name": "John",
  "email": "john@test.com"
}
```

| Column | Type |
|---|---|
| id | bigint |
| name | string |
| email | string |

> **Schema Location stores both versions.**

---

# Who Owns What?

| Component | Managed By |
|---|---|
| **Schema Location** | Auto Loader |
| **RocksDB** | Auto Loader |
| **Offsets** | Structured Streaming |
| **Commits** | Structured Streaming |
| **Sources** | Structured Streaming |
| **Checkpoint** | Structured Streaming |
| **Micro Batch** | Structured Streaming |

---

# Ultimate Interview Summary

| Component | One-Line Answer |
|---|---|
| **Schema Location** | Stores schema versions and schema evolution history. |
| **RocksDB** | Stores discovered and processed file metadata to avoid duplicate ingestion. |
| **Micro Batch** | Group of files processed in one trigger. |
| **Offsets** | Record what a batch is supposed to process. |
| **Commits** | Record which batches completed successfully. |
| **Sources** | Store source configuration and input metadata. |
| **Checkpoint** | Complete recovery package used to restart the stream without data loss. |

---

# ⭐ Golden Memory Trick

> **Schema Location** = What does data look like?  
>
> **RocksDB** = Have I seen this file before?  
>
> **Offset** = What should I process?  
>
> **Commit** = What did I successfully process?  
>
> **Sources** = Where is data coming from?  
>
> **Checkpoint** = How do I recover after failure?  
>
> **Micro Batch** = What am I processing right now?

# Auto Loader Schema Evolution - Complete Notes

## 1. Components Involved

| Component | Purpose | Stores |
|---|---|---|
| Schema Location | Source schema repository | Schema versions, columns, datatypes |
| Cached Schema (Memory) | Schema used by running stream | Current active schema |
| Auto Loader | Reads files | Compares file schema with cached schema |
| Delta Table | Target table | Table schema |
| mergeSchema | Delta schema evolution | Allows new columns in target table |

---

## 2. Initial Flow

### File 1

```json
{
"id": 1,
"name": "John"
}
```

### File 2

```json
{
"id": 2,
"name": "Mike"
}
```

**Auto Loader infers:**

```text
id
name
```

### Schema Stored

**Schema Location (Permanent Copy)**

**Version 1**

```text
id
name
```

**Cached Schema (Memory Copy)**

**When stream starts:**

```text
id
name
```

gets loaded into memory.

```text
Schema Location
↓
Load Schema
↓
Memory Cache
```

---

## 3. New File Arrives

### File 3

```json
{
"id": 3,
"name": "David",
"email": "d@test.com"
}
```

**File schema:**

```text
id
name
email
```

**New column:**

```text
email
```

---

## 4. What Auto Loader Does

Auto Loader compares:

| Cached Schema | File Schema |
|---|---|
| id | id |
| name | name |
| ❌ email missing | ✅ email exists |

Schema mismatch detected.

---

## 5. Auto Loader Updates Schema Location

Old Schema:

**Version 1**

```text
id
name
```

New Schema:

**Version 2**

```text
id
name
email
```

**Stored in:**

```text
schemaLocation
```

### Important Point

At this time:

**Schema Location**

```text
id
name
email
```

**Memory Cache**

```text
id
name
```

Still old.

---

## 6. Why Stream Fails

Auto Loader does NOT immediately change the schema already loaded into memory.

**The running query still uses:**

```text
id
name
```

**When processing File3:**

File

```text
id
name
email
```

**Cache**

```text
id
name
```

**Mismatch:**

```text
email column not found
```

**Result:**

```text
UnknownFieldException
or
Schema mismatch error
```

Stream stops.

---

## 7. Why Doesn't Auto Loader Refresh Cache Immediately?

Spark Structured Streaming works like:

```text
Start Stream
↓
Read Schema
↓
Create Execution Plan
↓
Run Micro Batches
```

The execution plan is already built.

Spark cannot safely modify the schema in the middle of a running query.

**Therefore:**

```text
Update schemaLocation
↓
Stop/Fail Query
↓
Restart Query
↓
Load Updated Schema
```

---

## 8. What Does Restart Mean?

Restart does NOT mean:

```text
Restart Cluster
```

Restart means:

```text
Stop Streaming Query
↓
Start Streaming Query Again
```

Example:

```python
query.stop()
```

Then start again.

---

## 9. After Restart

Stream starts again.

Reads latest schema from:

```text
schemaLocation
```

**Current schema:**

```text
id
name
email
```

Loads into memory.

**New Cache**

```text
id
name
email
```

**Now:**

| File Schema | Cached Schema |
|---|---|
| id | id |
| name | name |
| email | email |

Match ✅

Auto Loader succeeds.

---

## 10. Two Schema Checks Happen

This is a very important interview concept.

### Check 1: Auto Loader Schema Validation

**Auto Loader compares:**

```text
Incoming File Schema
VS
Cached Schema
```

**Example:**

```text
id
name
email
```

**VS**

```text
id
name
```

**Failure occurs.**

After restart:

```text
id
name
email

VS

id
name
email
```

Success.

### Check 2: Delta Table Schema Validation

Suppose target table:

```text
orders

id
name
```

**Incoming data:**

```text
id
name
email
```

Auto Loader is happy.

But Delta says:

```text
I don't have email column.
```

Write fails.

---

## 11. Delta Schema Evolution

**Enable:**

```python
.option("mergeSchema","true")
```

This allows Delta to evolve.

### Before

**Target table:**

```text
id
name
```

**Incoming Data**

```text
id
name
email
```

### After mergeSchema

**Target table:**

```text
id
name
email
```

Write succeeds.

---

## 12. Who Handles What?

| Layer | Responsibility |
|---|---|
| Auto Loader | Read new columns from source files |
| Schema Location | Store source schema history |
| Cached Schema | Active schema used by stream |
| Delta Table | Store target schema |
| mergeSchema | Add new columns to Delta table |

---

## 13. Autoloader vs Delta Schema Evolution

| Feature | Auto Loader | Delta Table |
|---|---|---|
| Purpose | Read new file columns | Update target table schema |
| Location | schemaLocation | Delta table metadata |
| Setting | cloudFiles.schemaEvolutionMode="addNewColumns" | mergeSchema=true |
| Handles | Source schema changes | Target schema changes |

---

## 14. Complete End-to-End Flow

```text
File1
(id,name)
↓
Infer Schema
↓
Schema Location
(id,name)
↓
Load Into Cache
(id,name)
↓
Write To Delta

Later:

File3
(id,name,email)
↓
Compare With Cache
(id,name)
↓
New Column Found
↓
Update schemaLocation
(id,name,email)
↓
Cache Still Old
(id,name)
↓
Mismatch
↓
Stream Stops
↓
Restart Query
↓
Reload Schema
(id,name,email)
↓
Auto Loader Success
↓
Check Delta Table
↓
Need mergeSchema=true
↓
Delta Adds email
↓
Write Success
```

## Interview Answer (Best Version)

Auto Loader maintains schema in two places: a permanent copy in schemaLocation and a cached copy in the running stream's memory. When a file arrives with a new column, Auto Loader detects the schema change and updates schemaLocation. However, the running query still uses the old cached schema, causing a schema mismatch and stream failure. After the streaming query restarts, it reloads the latest schema from schemaLocation and can read the new column. A second schema validation then occurs at the Delta table. If the target table does not contain the new column, mergeSchema=true is required so Delta can evolve its schema and successfully write the data.



## What is Schema Evolution Mode?

**Schema Evolution Mode** decides what Auto Loader should do when a **new column appears in incoming files**.

## Schema Evolution Modes

| Option | Description |
|---|---|
| **addNewColumns (default)** | Stream fails. New columns are added to the schema. Existing columns do not evolve data types. |
| **rescue** | Schema is never evolved and stream does not fail due to schema changes. All new columns are recorded in the rescued data column. |
| **failOnNewColumns** | Stream fails. Stream does not restart unless the provided schema is updated, or the offending data file is removed. |
| **none** | Does not evolve the schema. New columns are ignored, and data is not rescued unless the `rescuedDataColumn` option is set. Stream does not fail due to schema changes. |

---

%md
# Auto Loader Schema Evolution Modes

## 1. addNewColumns (Default)

```python
.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
```

### Existing Schema

```text
id
name
```

### New File

```text
id
name
email
```

### What Happens?

1. Detect new column (email)
2. Update schemaLocation
3. Stream fails
4. Restart query
5. Load updated schema
6. Continue processing

### Result

```text
Schema Updated = Yes
Stream Fails = Yes
New Column Added = Yes
```

### Easy Memory

```text
Fail
↓
Update Schema
↓
Restart
↓
Continue
```

---

## 2. rescue

```python
.option("cloudFiles.schemaEvolutionMode", "rescue")
```

### Existing Schema

```text
id
name
```

### New File

```text
id
name
email
```

### What Happens?

1. Do not evolve schema
2. Do not fail stream
3. Store unknown columns in _rescued_data

### Output

| id | name | _rescued_data |
|---|---|---|
| 1 | John | {"email":"john@test.com"} |

### Result

```text
Schema Updated = No
Stream Fails = No
New Column Saved = Yes
```

### Easy Memory

```text
Don't Fail
Don't Evolve
Rescue Data
```

---

## 3. failOnNewColumns

```python
.option("cloudFiles.schemaEvolutionMode", "failOnNewColumns")
```

### New File

```text
id
name
email
```

### What Happens?

1. Detect new column
2. Fail immediately
3. Do not update schema
4. Require manual action

### Result

```text
Schema Updated = No
Stream Fails = Yes
Manual Fix Needed = Yes
```

### Easy Memory

```text
Fail
No Evolution
Manual Fix
```

---

## 4. none

```python
.option("cloudFiles.schemaEvolutionMode", "none")
```

### New File

```text
id
name
email
```

### What Happens?

1. Ignore email
2. Process only known columns
3. No schema update
4. No failure

### Output

```text
id
name
```

### Result

```text
Schema Updated = No
Stream Fails = No
New Column = Ignored
```

### Easy Memory

```text
Ignore
Don't Fail
Don't Evolve
```

---

# Comparison

```text
addNewColumns
    -> Update schema + Restart

rescue
    -> Store unknown columns in _rescued_data

failOnNewColumns
    -> Fail immediately

none
    -> Ignore unknown columns
```

---

# If _rescued_data Already Exists In Source

### Source File

```json
{
"id": 1,
"name": "John",
"_rescued_data": "abc"
}
```

### Auto Loader

```python
.option("rescuedDataColumn", "_rescued_data")
```

### Problem

```text
Source column name = _rescued_data

Rescue column name = _rescued_data

Conflict
```

### Solution

Use another rescue column name.

```python
.option("rescuedDataColumn", "_extra_columns")
```

### Result

```text
Source Columns:
--------------
id
name
_rescued_data

Auto Loader Rescue Column:
--------------------------
_extra_columns

No conflict.
```

---

# Interview Question

## Can _rescued_data store multiple unknown columns?

**Yes.**

### Input

```json
{
"id":1,
"name":"John",
"email":"a@test.com",
"phone":"9999999999"
}
```

### Output

```text
_rescued_data
```

```json
{
  "email":"a@test.com",
  "phone":"9999999999"
}
```

---

## Can _rescued_data store nested JSON?

**Yes.**

### Input

```json
{
"id":1,
"address":{
"city":"Hyd",
"zip":"500001"
}
}
```

### Output

```json
{
"address":{
"city":"Hyd",
"zip":"500001"
}
}
```

**Nested structure is preserved.**

### > important

# 1. Get Storage Account Name and Access Key

In [0]:
adlsAccount = dbutils.secrets.get(
    scope="key-vault-secrets",
    key="adlsName"
)

adlsKey = dbutils.secrets.get(
    scope="key-vault-secrets",
    key="adlsAccessKey"
)


# 2. Configure ADLS Access
spark.conf.set(
    "fs.azure.account.key." + adlsAccount + ".dfs.core.windows.net",
    adlsKey
)

# 3. Define Container and Source Folder

In [0]:
container = "raw"
folder = "sales"

# 4. Create Source Location

In [0]:
location = f"abfss://{container}@{adlsAccount}.dfs.core.windows.net/{folder}/"


# 5. Read Source Files Using Auto Loader

In [0]:
df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .load(location)
)

### **Schema Options**


### 1. User Defined Schema

In [0]:
user_schema = "id INT, name STRING, age INT"

df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .schema(user_schema)
         .load(location)
)

### 2. Infer Schema

Usually Infer Schema is very expensive and not recommended, but not with Auto Loader. Auto Loader efficiently samples data to infer the schema and stores it under cloudFiles.schemaLocation in our storage.

Another option cloudFiles.inferColumnTypes determines the proper data type from our data.

In [0]:
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.schemaLocation", "/mnt/data/inferred_schema")
      .option("cloudFiles.inferColumnTypes", "true")
      .load(location))

### 3.Schema Hints

We can enforce a part of our schema using Schema Hints.

In this example, we make sure that emp_id is read as BIGINT and not INT.

In [0]:
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.schemaLocation", "/mnt/data/inferred_schema")
      .option("cloudFiles.inferColumnTypes", "true")
      .option("cloudFiles.schemaHints", "emp_id bigint")
      .load(location))

### USE NOTIFICATIONS

**File Notification Mode**

When you use:

Python
.option("cloudFiles.useNotifications", "true")

Auto Loader uses notifications to know when new files arrive.

**Simple Difference**

**Directory Listing:**

- Auto Loader
- ↓
- Checks/scans the source folder
- ↓
- Finds new files
- ↓
- Processes them

**File Notification:**


-  New file arrives
-  ↓
-  Notification is generated
-  ↓
-  Auto Loader receives the notification
-  ↓
-  Processes the new file

**Easy to remember**

Directory Listing = Auto Loader searches for new files.

File Notification = Auto Loader is notified when new files arrive.

Purpose: File notification mode avoids repeatedly listing a large source directory and is useful when there are many files.

In [0]:
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.useNotifications", "true")
      .schema(userDefinedSchema)
      .load(location))

### Define Queue Storage Options

In [0]:
cloudFilesConf = {
    "cloudFiles.subscriptionId": dbutils.secrets.get(
        scope="key-vault-secrets",
        key="sp-subscriptionId"
    ),

    "cloudFiles.connectionString": queue_sas,

    "cloudFiles.format": "avro",

    "cloudFiles.tenantId": dbutils.secrets.get(
        scope="key-vault-secrets",
        key="sp-tenantId"
    ),

    "cloudFiles.clientId": dbutils.secrets.get(
        scope="key-vault-secrets",
        key="sp-clientId"
    ),

    "cloudFiles.clientSecret": dbutils.secrets.get(
        scope="key-vault-secrets",
        key="sp-clientKey"
    ),

    "cloudFiles.resourceGroup": dbutils.secrets.get(
        scope="key-vault-secrets",
        key="sp-rgName"
    ),

    "cloudFiles.useNotifications": "true",
    "cloudFiles.includeExistingFiles": "true",
    "cloudFiles.validateOptions": "true",
}

%md

## 🔍 Step-by-Step Breakdown

### 1. Define Configuration Dictionary

`cloudFilesConf = { ... }`

- This dictionary holds all the necessary options for Auto Loader to connect to Azure Queue Storage securely and correctly.

---

### 2. Authentication and Access

These entries retrieve secrets from Azure Key Vault using Databricks' `dbutils.secrets.get()`:

| Key | Description |
|---|---|
| `cloudFiles.subscriptionId` | Azure subscription ID |
| `cloudFiles.tenantId` | Azure Active Directory tenant ID |
| `cloudFiles.clientId` | Service principal client ID |
| `cloudFiles.clientSecret` | Service principal secret |
| `cloudFiles.resourceGroup` | Azure resource group name |

These are used to authenticate the service principal that accesses the storage.

---

### 3. Connection String

`"cloudFiles.connectionString": queue_sas`

- `queue_sas` is a variable holding the Shared Access Signature (SAS) connection string for the Azure Queue.
- This allows Auto Loader to listen for file arrival events.

---

### 4. File Format

`"cloudFiles.format": "avro"`

- Specifies the format of the incoming files. In this case, it's Avro, a common format for structured data.

---

### 5. Notification Settings

`"cloudFiles.useNotifications": "true"`

- Enables event-based ingestion using Azure Queue Storage notifications.

`"cloudFiles.includeExistingFiles": "true"`

- Ensures that existing files in the storage are also processed, not just new ones.

`"cloudFiles.validateOptions": "true"`

- Validates the configuration options before starting ingestion.

---

## ✅ Summary

This setup allows Databricks Auto Loader to:

- Authenticate securely using secrets
- Connect to Azure Queue Storage
- Ingest Avro files using event notifications
- Include both new and existing files

---

## 🔁 1. `"cloudFiles.useNotifications": "true"`

### ✅ Purpose:

This enables event-driven ingestion using Azure Queue Storage notifications.

### 🔍 How It Works:

- Azure Event Grid monitors your storage account.
- When a new file is added, Event Grid sends a message to an Azure Queue.
- Databricks Auto Loader listens to this queue.
- When a message arrives, Auto Loader automatically ingests the file.

### 🧠 Why Use It:

- **Efficient:** Avoids scanning the entire directory.
- **Scalable:** Handles millions of files with low overhead.
- **Real-time:** Ingests files as soon as they arrive.

### ⚠️ Requirements:

- You must configure Event Grid, Queue Storage, and Storage Account properly.
- The queue must be accessible via a SAS token or service principal.

---

## 📂 2. `"cloudFiles.includeExistingFiles": "true"`

### ✅ Purpose:

Ensures that existing files in the storage location are also processed, not just new ones.

### 🔍 How It Works:

- When Auto Loader starts, it scans the directory once to find files that were already present.
- These files are then ingested along with new ones detected via notifications.

### 🧠 Why Use It:

- Useful when you're starting a pipeline and want to process historical data.
- Ensures no data is missed, especially during initial setup.

### ⚠️ Considerations:

- This initial scan can take time if the directory has many files.
- After the first run, only new files are processed via notifications.

---

## ✅ 3. `"cloudFiles.validateOptions": "true"`

### ✅ Purpose:

Validates the configuration options before starting ingestion.

### 🔍 How It Works:

- Auto Loader checks if all required options (like authentication, format, queue connection) are correctly set.
- If any option is invalid or missing, it throws an error before execution.

### 🧠 Why Use It:

- Prevents runtime failures due to misconfiguration.
- Helps catch issues early during development or deployment.

### ⚠️ Tip:

Always keep this set to `"true"` during development. You can disable it in production if you're confident in the setup.

### Read Files Using Auto Loader with cloudFilesConf

In [0]:
autoloader_df = (
    spark.readStream
         .format("cloudFiles")
         .options(**cloudFilesConf)
         .load(
             f"wasbs://landing@{adls_account}.blob.core.windows.net/"
             f"currents/dataexport.prod-02.AzureBlob.integration."
             f"12345/event_type={table}/"
         )
)

%md

## Auto Loader Common Options

| Option | Description |
|---|---|
| `cloudFiles.format` | It specifies the data format from the source path. For example, it takes `.json` for JSON files, `.csv` for CSV files, etc. |
| `cloudFiles.allowOverwrites` | With the default value set as `true`, this decides whether to permit input directory file changes to overwrite existing data. |
| `cloudFiles.schemaLocation` | This describes the location for storing the inferred schema along with associated changes. |
| `cloudFiles.schemaEvolutionMode` | Sets various modes for schema evolution, i.e., when new columns are detected in the data. |
| `cloudFiles.schemaHints` | This is the schema information of your data provided by you to Auto Loader. |
| `cloudFiles.inferColumnTypes` | With the initial value set as `false`, this checks whether to infer exact column types when leveraging schema inference. |
| `cloudFiles.includeExistingFiles` | Set to `true` by default, this checks whether to include existing files in the stream processing input path or to only handle the new files arriving after initial setup. |
| `cloudFiles.maxBytesPerTrigger` | This sets the maximum number of bytes Auto Loader will process in every trigger. |
| `cloudFiles.maxFileAge` | Used to set the duration for which the event is tracked for deduplication purposes. This is commonly used when rapidly ingesting millions of files per hour. |
| `cloudFiles.resourceTags` | Key-value pairs to help identify the right resources. |
| `cloudFiles.validateOptions` | This checks if the Auto Loader options discussed above are having a valid value or not. |
| `cloudFiles.backfillInterval` | 100% delivery of the file that has been uploaded is not guaranteed via the file notification mode. You can use backfills to ensure all the files get processed. This parameter decides the interval for triggering the backfills. |
| `cloudFiles.partitionColumns` | A comma-separated list of Hive-style partition columns that you would like inferred from the directory structure of the files. Hive-style partition columns are key-value pairs combined by an equality sign, such as `base-path/a=x/b=1/c=y/file.format`. |

### Read Data Using Wildcard

%md

## Wildcard Options

| Option | Description |
|:---:|---|
| `?` | Matches any single character |
| `*` | Matches zero or more characters |
| `[abc]` | Matches a single character from the set `{a,b,c}` |
| `[a-z]` | Matches a single character from the range `{a...z}` |
| `[^a]` | Matches a single character not in the specified set or range |
| `{ab,cd}` | Matches a string from the set `{ab, cd}` |
| `{ab,c{de,fh}}` | Matches a string from the set `{ab, cde, cfh}` |

In [0]:
df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", <format>) \
    .schema(schema) \
    .load("root_folder/*/files")


### WRITE DATA

In [0]:
query = df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(once=True) \
    .option("checkpointLocation", "/checkpoint/orders") \
    .start("/target/orders")